# Quick Start

Embed reaction SMARTS with the pretrained transformer — single reaction, batch, and pooling comparison.

> Run from the **project root** with `uv sync --no-dev`. The first call downloads and caches the medium model from the Hugging Face Hub automatically; a local checkpoint in `models/` (if present) is used instead.

In [ ]:
import numpy as np
from rxn_smarts_embeddings.predict import predict, load_embedder

## Single reaction

`predict` returns a 1-D array `(d_model,)` for a single string.

In [ ]:
smarts = "[C;H1:1]=[N;H0:2]>>[C;H1:1]-[N;H0:2]"

emb = predict(smarts)

print(f"shape : {emb.shape}")
print(f"dtype : {emb.dtype}")
print(f"norm  : {np.linalg.norm(emb):.4f}")
print(f"mean  : {emb.mean():.4f}  std: {emb.std():.4f}")

## Batch

`predict` returns `(N, d_model)` for a list.

In [ ]:
reactions = [
    "[C;H1:1]=[N;H0:2]>>[C;H1:1]-[N;H0:2]",
    "[C:1]-[O:2]>>[C:1]=[O:2]",
    "c1ccccc1>>c1cccnc1",
    "[C;H2:1]-[C;H2:2]>>[C;H1:1]=[C;H1:2]",
]

embs = predict(reactions)
print(f"shape: {embs.shape}")   # (4, 256)

## Reusing the loaded model

Loading weights takes a moment. Hold onto the `embedder` object to avoid reloading between calls. `pooling="mean"` is the default (validated as strictly better than CLS pooling).

In [ ]:
embedder = load_embedder()   # auto-discovers the latest checkpoint, pooling="mean" by default

# embed in separate chunks
e1 = embedder.embed(reactions[:2], batch_size=32)
e2 = embedder.embed(reactions[2:], batch_size=32)
all_embs = np.vstack([e1, e2])

print(f"combined shape: {all_embs.shape}")

## Pairwise cosine similarity

In [ ]:
import matplotlib.pyplot as plt

# normalise rows → dot product == cosine similarity
norms = np.linalg.norm(embs, axis=1, keepdims=True)
normed = embs / norms
sim = normed @ normed.T

labels = [
    "C=N → C-N",
    "C-O → C=O",
    "benzene → pyridine",
    "C-C → C=C",
]

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(labels)), labels, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(labels)), labels, fontsize=8)
plt.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title("Pairwise embedding similarity")
plt.tight_layout()
plt.show()

## CLS vs mean pooling

`cls` uses the `[BOS]` token (BERT-style); `mean` averages all non-padding token states.

In [ ]:
e_cls  = predict(reactions, pooling="cls")
e_mean = predict(reactions, pooling="mean")

def cos_sim(a, b):
    return float(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine similarity — first vs second reaction:")
print(f"  CLS  pooling: {cos_sim(e_cls[0],  e_cls[1]):.4f}")
print(f"  mean pooling: {cos_sim(e_mean[0], e_mean[1]):.4f}")